# 04_silver2gold

## Setup

In [ ]:
# Mount colab
if True:
    import os
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

# Paths
PROJECT_ROOT: str = f"/content/drive/MyDrive/colab_notebooks/projects/trial_translator_project/v1.2"

Mounted at /content/drive


In [ ]:
# Install reqs
if True:
    %pip install -r {PROJECT_ROOT + "/requirements.txt"}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.2/511.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.4 MB/s eta 

In [ ]:
# -- env libraries --
import os
import sys
import importlib
import torch
import time
import ast
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from dataclasses import dataclass, asdict, field
from typing import List
from IPython.display import Image
sns.set_style("darkgrid")

# -- hugging face --
from datasets import load_dataset, Dataset

# Add the PROJECT_ROOT directory to the Python path to import modules
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# -- project imports --
import src
from src.utils import push_and_tag, fix_data_format

# def reload function for imports
def reload_module(modules: list):
    for module in modules:
        importlib.reload(module)

# Perform initial reload/load to ensure everything is up-to-date
modules = [src, src.utils] # Add the top-level src module and src.utils to the list
reload_module(modules)

print("✅All relevant modules have been imported/reloaded.")

✅All relevant modules have been imported/reloaded.


In [ ]:
# Config

@dataclass
class EnvConfig:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    is_kaggle: bool = False
    requires_keys: bool = True

@dataclass
class DataConfig:
    load_from_hub: bool = True
    scored_silver_data_path_local: str = PROJECT_ROOT + "/outputs/scored_silver_generation_checkpoint.csv"
    scored_silver_data_path_hub: str = "clinical-policy-scored-silver-distill-cot"
    gold_data_path_local: str = PROJECT_ROOT + "/outputs/gold_checkpoint.csv"
    gold_data_path_hub: str = "clinical-policy-gold-distill-cot"
    dataset_version: str = "v1.0"
    push_to_hub: bool = True

@dataclass
# NOTE: Use default_factory for mutable default values to prevent unintended shared state.
class CFG:
    env: EnvConfig = field(default_factory=EnvConfig)
    data: DataConfig = field(default_factory=DataConfig)

    def __post_init__(self):
        """Runs automatically after the class is initialized."""
        # 1. Load keys (if necessary)
        if self.env.requires_keys:
            if self.env.is_kaggle:
                try:
                    from kaggle_secrets import UserSecretsClient
                    os.environ["REPO_ID"] = UserSecretsClient().get_secret("REPO_ID")
                    # os.environ["HUGGINGFACE_READ_TOKEN"] = UserSecretsClient().get_secret("HUGGINGFACE_READ_TOKEN")
                    os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
                    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
                    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
                    print("✅ All Kaggle Secrets loaded.")
                except Exception as e:
                    print(f"ℹ️ Kaggle secrets not found or an error occurred: {e}")
            else:
                try:
                    from google.colab import userdata
                    os.environ["REPO_ID"] = userdata.get("REPO_ID")
                    # os.environ["HUGGINGFACE_READ_TOKEN"] = userdata.get("HUGGINGFACE_READ_TOKEN")
                    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
                    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
                    os.environ["HF_TOKEN"] = userdata.get("HF_WRITE_TOKEN")
                    print("✅ All Colab Userdata Secrets loaded.")
                except Exception as e:
                    print(f"ℹ️ Colab secrets not found or an error occurred: {e}")

    def to_dict(self):
        return asdict(self)

In [ ]:
# Initialize config
config = CFG()
pprint(config.to_dict())

✅ All Colab Userdata Secrets loaded.
{'data': {'dataset_version': 'v1.0',
          'gold_data_path_hub': 'clinical-policy-gold-distill-cot',
          'gold_data_path_local': '/content/drive/MyDrive/colab_notebooks/projects/trial_translator_project/v1.2/outputs/gold_checkpoint.csv',
          'load_from_hub': True,
          'push_to_hub': True,
          'scored_silver_data_path_hub': 'clinical-policy-scored-silver-distill-cot',
          'scored_silver_data_path_local': '/content/drive/MyDrive/colab_notebooks/projects/trial_translator_project/v1.2/outputs/scored_silver_generation_checkpoint.csv'},
 'env': {'device': 'cpu',
         'is_kaggle': False,
         'requires_keys': True,
         'seed': 42}}


## Data

### Load data

In [ ]:
if config.data.load_from_hub:
    print("ℹ️ Loading data from hub 🤗.")
    full_repo_id = f"{os.environ['REPO_ID']}/{config.data.scored_silver_data_path_hub}"
    cot_ds = load_dataset(
        full_repo_id,
        split='train',
        revision="v1.0"
    )
else:
    print("ℹ️ Loading data from local.")
    cot_df = pd.read_csv(config.data.scored_silver_data_path_local)
    cot_ds = Dataset.from_pandas(cot_df)

display(cot_ds.to_pandas().head())
display(cot_ds.shape)

ℹ️ Loading data from hub 🤗.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/668 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/959k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120 [00:00<?, ? examples/s]

,nctId,keyword,prompt,document,target_audience,generations,model_names,distilabel_metadata,ratings,rationales,judge_model
0,NCT01633502,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Danish Cardiogenic Shock Trial\n\nSUMMA...,Policy Makers,['<think>\n* Key technical facts: Cardiogenic ...,"['gemini/gemma-3-27b-it', 'groq/meta-llama/lla...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 4]","['Text 1 follows all formatting rules, provide...",groq/openai/gpt-oss-120b
1,NCT01282203,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Volatile Induction and Ma...,Policy Makers,['<think>\n* Key technical facts: This is a po...,"['gemini/gemma-3-27b-it', 'groq/meta-llama/lla...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]","['Text 1 follows all formatting rules, provide...",groq/openai/gpt-oss-120b
2,NCT02436421,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Impact of Best Practice Alerts in Cardi...,Policy Makers,['<think>\n* Key technical facts: The study im...,"['gemini/gemma-3-27b-it', 'groq/meta-llama/lla...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]","['Text 1 follows all formatting rules, present...",groq/openai/gpt-oss-120b
3,NCT03935230,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Uncontrolled Donation After Circulatory...,Policy Makers,['<think>\n* Key technical facts: The document...,"['gemini/gemma-3-27b-it', 'groq/meta-llama/lla...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]",['Text 1 follows the required <think> tag form...,groq/openai/gpt-oss-120b
4,NCT02501681,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Different Priming Solutio...,Policy Makers,"[""<think>\n* Key technical facts: This is a si...","['gemini/gemma-3-27b-it', 'groq/meta-llama/lla...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]","['Text 1 follows all formatting rules, provide...",groq/openai/gpt-oss-120b


(120, 11)

In [ ]:
# Fix data formats
scored_silver_df = fix_data_format(cot_ds.to_pandas())
scored_silver_df.head()

ℹ️ Reformatting columns to Python objects...
  - Processing column: 'generations' (current type: object)
    --> Fixed type: <class 'list'>
  - Processing column: 'model_names' (current type: object)
    --> Fixed type: <class 'list'>
  - Processing column: 'ratings' (current type: object)
    --> Fixed type: <class 'list'>
  - Processing column: 'rationales' (current type: object)
    --> Fixed type: <class 'list'>
  - Processing column: 'distilabel_metadata' (current type: object)
    --> Fixed type: <class 'dict'>


,nctId,keyword,prompt,document,target_audience,generations,model_names,distilabel_metadata,ratings,rationales,judge_model
0,NCT01633502,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Danish Cardiogenic Shock Trial\n\nSUMMA...,Policy Makers,[<think>\n* Key technical facts: Cardiogenic s...,"[gemini/gemma-3-27b-it, groq/meta-llama/llama-...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 4]","[Text 1 follows all formatting rules, provides...",groq/openai/gpt-oss-120b
1,NCT01282203,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Volatile Induction and Ma...,Policy Makers,[<think>\n* Key technical facts: This is a pos...,"[gemini/gemma-3-27b-it, groq/meta-llama/llama-...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]","[Text 1 follows all formatting rules, provides...",groq/openai/gpt-oss-120b
2,NCT02436421,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Impact of Best Practice Alerts in Cardi...,Policy Makers,[<think>\n* Key technical facts: The study imp...,"[gemini/gemma-3-27b-it, groq/meta-llama/llama-...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]","[Text 1 follows all formatting rules, presents...",groq/openai/gpt-oss-120b
3,NCT03935230,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Uncontrolled Donation After Circulatory...,Policy Makers,[<think>\n* Key technical facts: The document ...,"[gemini/gemma-3-27b-it, groq/meta-llama/llama-...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]",[Text 1 follows the required <think> tag forma...,groq/openai/gpt-oss-120b
4,NCT02501681,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Different Priming Solutio...,Policy Makers,[<think>\n* Key technical facts: This is a sin...,"[gemini/gemma-3-27b-it, groq/meta-llama/llama-...","{'raw_output_ultrafeedback_ranking': '{""rating...","[5, 5]","[Text 1 follows all formatting rules, provides...",groq/openai/gpt-oss-120b


### Explode dataset

In [ ]:
COLS_TO_EXPLODE = ["generations", "model_names", "ratings", "rationales"]
exploded_df = scored_silver_df.explode(COLS_TO_EXPLODE)
exploded_df.head()

,nctId,keyword,prompt,document,target_audience,generations,model_names,distilabel_metadata,ratings,rationales,judge_model
0,NCT01633502,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Danish Cardiogenic Shock Trial\n\nSUMMA...,Policy Makers,<think>\n* Key technical facts: Cardiogenic sh...,gemini/gemma-3-27b-it,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,"Text 1 follows all formatting rules, provides ...",groq/openai/gpt-oss-120b
0,NCT01633502,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Danish Cardiogenic Shock Trial\n\nSUMMA...,Policy Makers,<think>\n- **Key Technical Facts**: \n - Card...,groq/meta-llama/llama-4-scout-17b-16e-instruct,"{'raw_output_ultrafeedback_ranking': '{""rating...",4,Text 2 mostly follows the instructions and is ...,groq/openai/gpt-oss-120b
1,NCT01282203,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Volatile Induction and Ma...,Policy Makers,<think>\n* Key technical facts: This is a post...,gemini/gemma-3-27b-it,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,"Text 1 follows all formatting rules, provides ...",groq/openai/gpt-oss-120b
1,NCT01282203,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Volatile Induction and Ma...,Policy Makers,<think>\n* Key technical facts:\n - The study...,groq/meta-llama/llama-4-scout-17b-16e-instruct,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,Text 2 also adheres to the required tags and f...,groq/openai/gpt-oss-120b
2,NCT02436421,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Impact of Best Practice Alerts in Cardi...,Policy Makers,<think>\n* Key technical facts: The study impl...,gemini/gemma-3-27b-it,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,"Text 1 follows all formatting rules, presents ...",groq/openai/gpt-oss-120b


In [ ]:
# Check statistics by model
exploded_df.groupby('model_names')['ratings'].value_counts()

model_names                                     ratings
gemini/gemma-3-27b-it                           5          106
                                                4           13
                                                3            1
groq/meta-llama/llama-4-scout-17b-16e-instruct  5           71
                                                4           37
                                                3            9
                                                2            3
Name: count, dtype: int64

### **Model Performance Comparative Analysis & Quality Distribution**

The LLM-as-a-Judge evaluation reveals a distinct performance gap between the two generator architectures.

#### **1. Model Performance Distribution**
| Model | Rating 5 (Elite) | Rating 4 (Good) | Rating 3-2 (Suboptimal) |
| :--- | :--- | :--- | :--- |
| **Gemma-3-27B** | **106** | 13 | 1 |
| **Llama-4-Scout** | 71 | 37 | 12 |

#### **2. Analytical Insights**
* **Consistency:** `Gemma-3-27B` demonstrates superior instruction-following and policy adherence, with **88.3%** of responses achieving a perfect score.
* **Variability:** `Llama-4-Scout` shows a wider distribution. While still highly capable (59% of perfect scores), the presence of 2-star ratings suggests occasional "hallucinations" or deviations from the clinical constraints provided in the prompt.
* **Note:** A significant portion of the 5-star ratings are overlapping. These "Consensus 5s" represent the highest-confidence data points in our corpus.

#### **3. Refined Gold Extraction Logic**
To maximize the "Signal-to-Noise" ratio for fine-tuning, we apply the following filtration heuristic:
1. **Selection:** Inclusion is restricted to responses with a **Rating $\ge 4$**.
2. **Preference:** If a prompt has both a 5-star and a 4-star response, only the 5-star response is promoted to Gold.
3. **Multi-Entry:** If both models achieve a 5-star rating, both are included to provide the student model with diverse high-quality linguistic patterns for the same clinical reasoning path.

In [ ]:
# Selection: Include responses with a Rating >= 4
filtered_df = exploded_df.query('ratings >= 4').copy()

# Find the maximum rating for each nctId within the filtered data
max_ratings_per_nctid = filtered_df.groupby('nctId')['ratings'].transform('max')

# Preference/Multi-Entry: Keep only the highest rated if there's a difference (e.g., 5 over 4),
# and keep all if they are equally highest (e.g., two 5s).
gold_df = filtered_df.query('ratings == @max_ratings_per_nctid')

print("Shape of gold_df:", gold_df.shape)
display(gold_df.head())

# Optional: Check counts again for the gold_df
print("\nGold Data Ratings Distribution:")
display(gold_df.groupby('model_names')['ratings'].value_counts())

Shape of gold_df: (191, 11)


,nctId,keyword,prompt,document,target_audience,generations,model_names,distilabel_metadata,ratings,rationales,judge_model
0,NCT01633502,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Danish Cardiogenic Shock Trial\n\nSUMMA...,Policy Makers,<think>\n* Key technical facts: Cardiogenic sh...,gemini/gemma-3-27b-it,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,"Text 1 follows all formatting rules, provides ...",groq/openai/gpt-oss-120b
1,NCT01282203,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Volatile Induction and Ma...,Policy Makers,<think>\n* Key technical facts: This is a post...,gemini/gemma-3-27b-it,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,"Text 1 follows all formatting rules, provides ...",groq/openai/gpt-oss-120b
1,NCT01282203,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Evaluation of Volatile Induction and Ma...,Policy Makers,<think>\n* Key technical facts:\n - The study...,groq/meta-llama/llama-4-scout-17b-16e-instruct,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,Text 2 also adheres to the required tags and f...,groq/openai/gpt-oss-120b
2,NCT02436421,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Impact of Best Practice Alerts in Cardi...,Policy Makers,<think>\n* Key technical facts: The study impl...,gemini/gemma-3-27b-it,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,"Text 1 follows all formatting rules, presents ...",groq/openai/gpt-oss-120b
2,NCT02436421,Cardiovascular,INSTRUCTION: \n You are an expert technical...,TITLE: Impact of Best Practice Alerts in Cardi...,Policy Makers,<think>\n* Key technical facts:\n - Two Best ...,groq/meta-llama/llama-4-scout-17b-16e-instruct,"{'raw_output_ultrafeedback_ranking': '{""rating...",5,Text 2 also adheres to the required tags and b...,groq/openai/gpt-oss-120b



Gold Data Ratings Distribution:


model_names                                     ratings
gemini/gemma-3-27b-it                           5          106
                                                4            8
groq/meta-llama/llama-4-scout-17b-16e-instruct  5           71
                                                4            6
Name: count, dtype: int64

In [ ]:
# Push to hub
if config.data.push_to_hub:
    full_repo_id = f"{os.environ['REPO_ID']}/{config.data.gold_data_path_hub}"
    push_and_tag(
        Dataset.from_pandas(gold_df),
        full_repo_id,
        commit_msg="Uploading (gold) synthetic distilled CoT dataset.",
        artifact_type="dataset",
        version_tag=config.data.dataset_version
    )

ℹ️Pushing to Francesco-A/clinical-policy-gold-distill-cot


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  60%|#####9    |  540kB /  903kB            

Creating tag v1.0
✅Done!


In [ ]:
test_ds = load_dataset(full_repo_id)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/903k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/191 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['nctId', 'keyword', 'prompt', 'document', 'target_audience', 'generations', 'model_names', 'distilabel_metadata', 'ratings', 'rationales', 'judge_model', '__index_level_0__'],
        num_rows: 191
    })
})

## Requirements

In [ ]:
!pip freeze

absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.5
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.13.0
anywidget==0.9.21
apsw==3.51.3.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.3.30.0.54.34
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.6.9
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.38.0
bigquery-magics==0.12.2
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.1.2
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.2.25
c

```markdown
absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.5
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.13.0
anywidget==0.9.21
apsw==3.51.3.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.3.30.0.54.34
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.6.9
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.38.0
bigquery-magics==0.12.2
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.1.2
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.2.25
cffi==2.0.0
chardet==5.2.0
charset-normalizer==3.4.6
clarabel==0.11.1
click==8.1.8
click-plugins==1.1.1.2
cligj==0.7.2
cloudpathlib==0.23.0
cloudpickle==3.1.2
cmake==3.31.10
cmdstanpy==1.3.0
colorcet==3.1.0
colorlover==0.3.0
colour==0.1.5
community==1.0.0b1
confection==1.3.3
cons==0.4.7
contourpy==1.3.3
cramjam==2.11.0
cryptography==43.0.3
cufflinks==0.17.3
curl_cffi==0.14.0
cvxopt==1.3.2
cvxpy==1.6.7
cycler==0.12.1
cyipopt==1.5.0
cymem==2.0.13
Cython==3.0.12
dask==2026.3.0
dataproc-spark-connect==1.0.2
datasets==4.0.0
db-dtypes==1.5.1
dbus-python==1.2.18
debugpy==1.8.15
decorator==4.4.2
defusedxml==0.7.1
deprecation==2.1.0
diffusers==0.37.1
dill==0.3.8
diskcache==5.6.3
distilabel==1.5.3
distro==1.9.0
dlib==19.24.6
dm-tree==0.1.10
docstring_parser==0.17.0
docutils==0.21.2
dopamine_rl==4.1.2
duckdb==1.3.2
earthengine-api==1.5.24
easydict==1.13
editdistance==0.8.1
eerepr==0.1.2
einops==0.8.2
en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl#sha256=1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85
entrypoints==0.4
esda==2.9.0
et_xmlfile==2.0.0
etils==1.14.0
etuples==0.3.10
Farama-Notifications==0.0.4
fastai==2.8.7
fastapi==0.135.2
fastcore==1.12.33
fastdownload==0.0.7
fastjsonschema==2.21.2
fastlite==0.2.4
fastprogress==1.1.5
fasttransform==0.0.2
fastuuid==0.14.0
ffmpy==1.0.0
filelock==3.25.2
fiona==1.10.1
firebase-admin==6.9.0
Flask==3.1.3
flatbuffers==25.12.19
flax==0.11.2
folium==0.20.0
fonttools==4.62.1
fqdn==1.5.1
frozendict==2.4.7
frozenlist==1.8.0
fsspec==2025.3.0
future==1.0.0
gast==0.7.0
gcsfs==2025.3.0
GDAL==3.8.4
gdown==5.2.1
geemap==0.35.3
geocoder==1.38.1
geographiclib==2.1
geopandas==1.1.3
geopy==2.4.1
giddy==2.3.6
gin-config==0.5.0
gitdb==4.0.12
GitPython==3.1.46
glob2==0.7
google==3.0.0
google-adk==1.28.0
google-ai-generativelanguage==0.6.15
google-api-core==2.30.1
google-api-python-client==2.193.0
google-auth==2.47.0
google-auth-httplib2==0.3.1
google-auth-oauthlib==1.3.1
google-cloud-aiplatform==1.143.0
google-cloud-appengine-logging==1.9.0
google-cloud-audit-log==0.5.0
google-cloud-bigquery==3.41.0
google-cloud-bigquery-connection==1.21.0
google-cloud-bigquery-storage==2.37.0
google-cloud-bigtable==2.35.0
google-cloud-core==2.5.1
google-cloud-dataplex==2.18.0
google-cloud-dataproc==5.26.0
google-cloud-datastore==2.24.0
google-cloud-discoveryengine==0.13.12
google-cloud-firestore==2.26.0
google-cloud-functions==1.23.0
google-cloud-iam==2.22.0
google-cloud-language==2.20.0
google-cloud-logging==3.15.0
google-cloud-monitoring==2.30.0
google-cloud-pubsub==2.36.0
google-cloud-resource-manager==1.17.0
google-cloud-secret-manager==2.27.0
google-cloud-spanner==3.63.0
google-cloud-speech==2.38.0
google-cloud-storage==3.10.1
google-cloud-trace==1.19.0
google-cloud-translate==3.25.0
google-colab @ file:///colabtools/dist/google_colab-1.0.0.tar.gz
google-crc32c==1.8.0
google-genai==1.68.0
google-generativeai==0.8.6
google-pasta==0.2.0
google-resumable-media==2.8.2
googleapis-common-protos==1.73.1
googledrivedownloader==1.1.0
gradio==5.50.0
gradio_client==1.14.0
grain==0.2.16
graphviz==0.21
greenlet==3.3.2
groovy==0.1.2
groq==1.1.2
grpc-google-iam-v1==0.14.3
grpc-interceptor==0.15.4
grpcio==1.80.0
grpcio-status==1.71.2
grpclib==0.4.9
gspread==6.2.1
gspread-dataframe==4.0.0
gym==0.25.2
gym-notices==0.1.0
gymnasium==1.2.3
h11==0.16.0
h2==4.3.0
h5netcdf==1.8.1
h5py==3.16.0
hdbscan==0.8.42
hf-xet==1.4.2
highspy==1.13.1
holidays==0.93
holoviews==1.22.1
hpack==4.1.0
html5lib==1.1
httpcore==1.0.9
httpimport==1.4.1
httplib2==0.31.2
httptools==0.7.1
httpx==0.28.1
httpx-sse==0.4.3
huggingface_hub==1.6.0
humanize==4.15.0
hyperframe==6.1.0
hyperopt==0.2.7
ibis-framework==9.5.0
idna==3.11
ImageIO==2.37.3
imageio-ffmpeg==0.6.0
imagesize==2.0.0
imbalanced-learn==0.14.1
immutabledict==4.3.1
importlib_metadata==8.5.0
importlib_resources==6.5.2
imutils==0.5.4
inequality==1.1.2
inflect==7.5.0
iniconfig==2.3.0
instructor==1.14.5
intel-cmplr-lib-ur==2025.3.3
intel-openmp==2025.3.3
ipyevents==2.0.4
ipyfilechooser==0.6.0
ipykernel==6.17.1
ipyleaflet==0.20.0
ipyparallel==8.8.0
ipython==7.34.0
ipython-genutils==0.2.0
ipython-sql==0.5.0
ipytree==0.2.2
ipywidgets==7.7.1
isoduration==20.11.0
itsdangerous==2.2.0
jaraco.classes==3.4.0
jaraco.context==6.1.2
jaraco.functools==4.4.0
jax==0.7.2
jaxlib==0.7.2
jeepney==0.9.0
jieba==0.42.1
Jinja2==3.1.6
jiter==0.11.1
joblib==1.5.3
jsonpatch==1.33
jsonpickle==4.1.1
jsonpointer==3.1.1
jsonschema==4.23.0
jsonschema-specifications==2025.9.1
jupyter-console==6.6.3
jupyter-events==0.12.0
jupyter-leaflet==0.20.0
jupyter_client==7.4.9
jupyter_core==5.9.1
jupyter_kernel_gateway @ git+https://github.com/googlecolab/kernel_gateway@b134e9945df25c2dcb98ade9129399be10788671
jupyter_server==2.14.0
jupyter_server_terminals==0.5.4
jupyterlab_pygments==0.3.0
jupyterlab_widgets==3.0.16
jupytext==1.19.1
kaggle==2.0.0
kagglehub==1.0.0
kagglesdk==0.1.16
keras==3.13.2
keras-hub==0.26.0
keras-nlp==0.26.0
keyring==25.7.0
keyrings.google-artifactregistry-auth==1.1.2
kiwisolver==1.5.0
langchain==1.2.14
langchain-core==1.2.23
langgraph==1.1.4
langgraph-checkpoint==4.0.1
langgraph-prebuilt==1.0.8
langgraph-sdk==0.3.12
langsmith==0.7.23
lark==1.3.1
launchpadlib==1.10.16
lazr.restfulclient==0.14.4
lazr.uri==1.0.6
lazy-loader==0.5
libclang==18.1.1
libpysal==4.14.1
librosa==0.11.0
lightgbm==4.6.0
linkify-it-py==2.1.0
litellm==1.83.4
llvmlite==0.43.0
locket==1.0.0
logical-unification==0.4.7
lxml==6.0.2
Mako==1.3.10
mapclassify==2.10.0
Markdown==3.10.2
markdown-it-py==4.0.0
MarkupSafe==3.0.3
matplotlib==3.10.0
matplotlib-inline==0.2.1
matplotlib-venn==1.1.2
mcp==1.26.0
mdit-py-plugins==0.5.0
mdurl==0.1.2
mgwr==2.2.1
miniKanren==1.0.5
missingno==0.5.2
mistune==3.2.0
mizani==0.13.5
mkl==2025.3.1
ml_dtypes==0.5.4
mlxtend==0.23.4
mmh3==5.2.1
momepy==0.11.0
more-itertools==10.8.0
moviepy==1.0.3
mpmath==1.3.0
msgpack==1.1.2
multidict==6.7.1
multipledispatch==1.0.0
multiprocess==0.70.16
multitasking==0.0.12
murmurhash==1.0.15
music21==9.9.1
namex==0.1.0
narwhals==2.18.1
natsort==8.4.0
nbclassic==1.3.3
nbclient==0.10.4
nbconvert==7.17.0
nbformat==5.10.4
ndindex==1.10.1
nest-asyncio==1.6.0
networkx==3.6.1
nibabel==5.4.2
nltk==3.9.1
notebook==6.5.7
notebook_shim==0.2.4
numba==0.60.0
numexpr==2.14.1
numpy==2.0.2
nvidia-nccl-cu12==2.29.7
oauth2client==4.1.3
oauthlib==3.3.1
omegaconf==2.3.0
onemkl-license==2025.3.1
openai==2.30.0
opencv-contrib-python==4.13.0.92
opencv-python==4.13.0.92
opencv-python-headless==4.13.0.92
openpyxl==3.1.5
opentelemetry-api==1.38.0
opentelemetry-exporter-gcp-logging==1.11.0a0
opentelemetry-exporter-gcp-monitoring==1.11.0a0
opentelemetry-exporter-gcp-trace==1.11.0
opentelemetry-exporter-otlp-proto-common==1.38.0
opentelemetry-exporter-otlp-proto-http==1.38.0
opentelemetry-proto==1.38.0
opentelemetry-resourcedetector-gcp==1.11.0a0
opentelemetry-sdk==1.38.0
opentelemetry-semantic-conventions==0.59b0
opt_einsum==3.4.0
optax==0.2.8
optree==0.19.0
orbax-checkpoint==0.11.33
orjson==3.11.8
ormsgpack==1.12.2
osqp==1.1.1
overrides==7.7.0
packaging==26.0
pandas==2.2.2
pandas-datareader==0.10.0
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
pandocfilters==1.5.1
panel==1.8.10
param==2.3.3
parso==0.8.6
parsy==2.2
partd==1.4.2
pathlib_abc==0.5.2
patsy==1.0.2
peewee==4.0.3
peft==0.18.1
pexpect==4.9.0
pickleshare==0.7.5
pillow==11.3.0
platformdirs==4.9.4
plotly==5.24.1
plotnine==0.14.5
pluggy==1.6.0
plum-dispatch==2.8.0
pointpats==2.5.5
polars==1.35.2
polars-runtime-32==1.35.2
pooch==1.9.0
portalocker==3.2.0
portpicker==1.5.2
preshed==3.0.13
prettytable==3.17.0
proglog==0.1.12
progressbar2==4.5.0
prometheus_client==0.24.1
promise==2.3
prompt_toolkit==3.0.52
propcache==0.4.1
prophet==1.3.0
proto-plus==1.27.2
protobuf==5.29.6
psutil==5.9.5
psycopg2==2.9.11
psygnal==0.15.1
ptyprocess==0.7.0
PuLP==3.3.0
py-cpuinfo==9.0.0
py4j==0.10.9.9
pyarrow==18.1.0
pyasn1==0.6.3
pyasn1_modules==0.4.2
pycairo==1.29.0
pycocotools==2.0.11
pycparser==3.0
pycryptodomex==3.23.0
pydantic==2.12.5
pydantic-settings==2.13.1
pydantic_core==2.41.5
pydata-google-auth==1.9.1
pydot==4.0.1
pydotplus==2.0.2
PyDrive2==1.21.3
pydub==0.25.1
pyerfa==2.0.1.5
pygame==2.6.1
pygit2==1.19.2
Pygments==2.20.0
PyGObject==3.48.2
pyiceberg==0.11.1
PyJWT==2.12.1
pymc==5.28.3
pynndescent==0.6.0
pyogrio==0.12.1
pyomo==6.10.0
PyOpenGL==3.1.10
pyOpenSSL==24.2.1
pyparsing==3.3.2
pyperclip==1.11.0
pyproj==3.7.2
pyroaring==1.0.4
pysal==25.7
pyshp==3.0.3
PySocks==1.7.1
pyspark==4.0.2
pytensor==2.38.2
pytest==8.4.2
python-apt==0.0.0
python-box==7.4.1
python-dateutil==2.9.0.post0
python-dotenv==1.0.1
python-fasthtml==0.12.50
python-json-logger==4.1.0
python-louvain==0.16
python-multipart==0.0.22
python-slugify==8.0.4
python-snappy==0.7.3
python-utils==3.9.1
pytz==2025.2
pyviz_comms==3.0.6
PyWavelets==1.9.0
PyYAML==6.0.3
pyzmq==26.2.1
quantecon==0.11.1
rasterio==1.5.0
rasterstats==0.20.0
ratelim==0.1.6
referencing==0.37.0
regex==2025.11.3
requests==2.32.4
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0
requirements-parser==0.9.0
rfc3339-validator==0.1.4
rfc3986-validator==0.1.1
rfc3987-syntax==1.1.0
rich==13.9.4
roman-numerals==4.1.0
roman-numerals-py==4.1.0
rpds-py==0.30.0
rpy2==3.5.17
rsa==4.9.1
rtree==1.4.1
ruff==0.15.8
safehttpx==0.1.7
safetensors==0.7.0
scikit-image==0.25.2
scikit-learn==1.6.1
scipy==1.16.3
scooby==0.11.0
scs==3.2.11
seaborn==0.13.2
SecretStorage==3.5.0
segregation==2.5.4
semantic-version==2.10.0
Send2Trash==2.1.0
sentence-transformers==5.3.0
sentencepiece==0.2.1
sentry-sdk==2.57.0
setuptools==75.2.0
shap==0.51.0
shapely==2.1.2
shellingham==1.5.4
simple-parsing==0.1.8
simplejson==3.20.2
simsimd==6.5.16
six==1.17.0
sklearn-compat==0.1.5
sklearn-pandas==2.2.0
slicer==0.0.8
smart_open==7.5.1
smmap==5.0.3
sniffio==1.3.1
snowballstemmer==3.0.1
soundfile==0.13.1
soupsieve==2.8.3
soxr==1.0.0
spacy==3.8.14
spacy-legacy==3.0.12
spacy-loggers==1.0.5
spaghetti==1.7.6
spanner-graph-notebook==1.1.8
spglm==1.1.0
Sphinx==8.2.3
sphinxcontrib-applehelp==2.0.0
sphinxcontrib-devhelp==2.0.0
sphinxcontrib-htmlhelp==2.1.0
sphinxcontrib-jsmath==1.0.1
sphinxcontrib-qthelp==2.0.0
sphinxcontrib-serializinghtml==2.0.0
spint==1.0.7
splot==1.1.7
spopt==0.7.0
spreg==1.9.0
SQLAlchemy==2.0.48
sqlalchemy-spanner==1.17.3
sqlglot==25.20.2
sqlparse==0.5.5
srsly==2.5.3
sse-starlette==3.3.4
stanio==0.5.1
starlette==0.52.1
statsmodels==0.14.6
strictyaml==1.7.3
stringzilla==4.6.0
stumpy==1.13.0
sympy==1.14.0
tables==3.10.2
tabulate==0.9.0
tbb==2022.3.1
tblib==3.2.2
tcmlib==1.4.1
tenacity==9.1.4
tensorboard==2.19.0
tensorboard-data-server==0.7.2
tensorflow==2.19.0
tensorflow-datasets==4.9.9
tensorflow-hub==0.16.1
tensorflow-metadata==1.17.3
tensorflow-probability==0.25.0
tensorflow-text==2.19.0
tensorflow_decision_forests==1.12.0
tensorstore==0.1.82
termcolor==3.3.0
terminado==0.18.1
text-unidecode==1.3
textblob==0.19.0
tf-slim==1.1.0
tf_keras==2.19.0
thinc==8.3.13
threadpoolctl==3.6.0
tifffile==2026.3.3
tiktoken==0.12.0
timm==1.0.26
tinycss2==1.4.0
tobler==0.13.0
tokenizers==0.22.2
toml==0.10.2
tomlkit==0.13.3
toolz==0.12.1
torch @ https://download.pytorch.org/whl/cpu/torch-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
torchao==0.10.0
torchaudio @ https://download.pytorch.org/whl/cpu/torchaudio-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
torchcodec @ https://download.pytorch.org/whl/cpu/torchcodec-0.10.0-cp312-cp312-manylinux_2_28_x86_64.whl
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision @ https://download.pytorch.org/whl/cpu/torchvision-0.25.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
tornado==6.5.1
tqdm==4.67.3
traitlets==5.7.1
traittypes==0.2.3
transformers==5.0.0
treescope==0.1.10
tsfresh==0.21.1
tweepy==4.16.0
typeguard==4.5.1
typer==0.23.1
typer-slim==0.23.1
types-pytz==2026.1.1.20260304
types-setuptools==82.0.0.20260402
typing-inspection==0.4.2
typing_extensions==4.15.0
tzdata==2025.3
tzlocal==5.3.1
uc-micro-py==2.0.0
umap-learn==0.5.11
umf==1.0.3
universal_pathlib==0.3.10
uri-template==1.3.0
uritemplate==4.2.0
urllib3==2.5.0
uuid_utils==0.14.1
uvicorn==0.42.0
uvloop==0.22.1
vega-datasets==0.9.0
wadllib==1.3.6
wandb==0.25.1
wasabi==1.1.3
watchdog==6.0.0
watchfiles==1.1.1
wcwidth==0.6.0
weasel==1.0.0
webcolors==25.10.0
webencodings==0.5.1
websocket-client==1.9.0
websockets==15.0.1
Werkzeug==3.1.7
wheel==0.46.3
widgetsnbextension==3.6.10
wordcloud==1.9.6
wrapt==2.1.2
wurlitzer==3.1.1
xarray==2025.12.0
xarray-einstats==0.10.0
xgboost==3.2.0
xlrd==2.0.2
xxhash==3.6.0
xyzservices==2026.3.0
yarl==1.23.0
ydf==0.15.0
yellowbrick==1.5
yfinance==0.2.66
zipp==3.23.0
zstandard==0.25.0
```

## Danger zone

In [ ]:
if False:
    # Delete and push new version
    from huggingface_hub import HfApi
    api = HfApi()
    full_repo_id = f"{os.environ['REPO_ID']}/{config.data.gold_data_path_hub}"
    api.delete_tag(
        repo_id=full_repo_id,
        tag=config.data.dataset_version,
        repo_type="dataset"
    )

    push_and_tag(
        Dataset.from_pandas(temp_df),
        full_repo_id,
        commit_msg="Uploading (gold) synthetic distilled CoT dataset.",
        artifact_type="dataset",
        version_tag=config.data.dataset_version
    )